# Molecule Lab

A dozen well-known drugs, drawn and measured with **RDKit**, on the booth's **conda** kernel.

RDKit is a C++ cheminformatics toolkit with Python bindings: exactly the kind of native stack conda
exists for. The booth installed it from conda-forge at build time (`install conda rdkit pandas matplotlib`)
and registered the conda environment as its own Jupyter kernel, **Python 3.x (conda)**, which is what this
notebook runs on. JupyterLab's stock *Python (venv)* kernel would not see RDKit at all.

*Run → Run All Cells* to fill in every output.

## 1. Which Python is this?

The interpreter lives under `/opt/conda-envs/`, the environment `conda--setup.sh` created, and `conda list`
shows RDKit came from conda-forge.

In [ ]:
import sys
import rdkit

print("Python:     ", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("RDKit:      ", rdkit.__version__)

In [ ]:
!conda list -p {sys.prefix} '^(rdkit|pandas|matplotlib)$'

## 2. The shelf

`src/molecules.py` holds the molecules (as SMILES) and the descriptor code. The command line uses the same
module: `just run caffeine`.

In [ ]:
sys.path.insert(0, "src")

import pandas as pd
from molecules import DRUGS, properties, to_mol

mols = {name: to_mol(name)[1] for name in DRUGS}
table = pd.DataFrame({name: properties(mol) for name, mol in mols.items()}).T
table = table.astype({"mol_weight": float, "logp": float, "tpsa": float})
table.round(2)

## 3. What they look like

RDKit draws 2-D structures straight from SMILES; the grid renders inline as SVG.

In [ ]:
from rdkit.Chem import Draw

Draw.MolsToGridImage(
    list(mols.values()),
    legends=list(mols.keys()),
    molsPerRow=4,
    subImgSize=(240, 200),
    useSVG=True,
)

## 4. Lipinski's rule of five

An orally active drug usually has molecular weight ≤ 500, logP ≤ 5, ≤ 5 H-bond donors and ≤ 10 acceptors,
with one violation tolerated. Most of the shelf passes. Two do not: rifampicin, a large antibiotic, and
atorvastatin, a statin taken as a pill by millions every day. The rule is a rule of thumb.

In [ ]:
table[["formula", "mol_weight", "logp", "h_donors", "h_acceptors", "lipinski_violations", "lipinski_pass"]] \
    .sort_values("mol_weight") \
    .round(2)

In [ ]:
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.add_patch(Rectangle((0, -2), 500, 7, color="tab:green", alpha=0.08, label="MW ≤ 500 and logP ≤ 5"))
ax.axvline(500, color="tab:green", linestyle="--", linewidth=1)
ax.axhline(5, color="tab:green", linestyle="--", linewidth=1)

colors = table["lipinski_pass"].map({True: "tab:blue", False: "tab:red"})
ax.scatter(table["mol_weight"], table["logp"], s=table["tpsa"] * 2, c=colors, alpha=0.7, edgecolor="black")

# A few molecules sit almost on top of each other; nudge their labels apart.
nudge = {"paracetamol": (-60, 8), "serotonin": (6, 10), "aspirin": (6, -12), "metformin": (-58, -4)}
for name, row in table.iterrows():
    ax.annotate(name, (row["mol_weight"], row["logp"]), xytext=nudge.get(name, (6, 4)),
                textcoords="offset points", fontsize=9)

ax.set_xlim(0, 900)
ax.set_ylim(-2, 7)
ax.set_xlabel("Molecular weight (g/mol)")
ax.set_ylabel("logP (Crippen)")
ax.set_title("Weight vs. lipophilicity  (bubble = polar surface area, red = fails the rule)")
ax.legend(loc="upper left")
plt.show()

## 5. Substructure search

Which molecules contain a carboxylic acid? RDKit matches a SMARTS pattern and highlights the atoms.

In [ ]:
from rdkit import Chem

pattern = Chem.MolFromSmarts("C(=O)[OH]")
hits = {name: mol for name, mol in mols.items() if mol.HasSubstructMatch(pattern)}

Draw.MolsToGridImage(
    list(hits.values()),
    legends=list(hits.keys()),
    highlightAtomLists=[list(mol.GetSubstructMatch(pattern)) for mol in hits.values()],
    molsPerRow=4,
    subImgSize=(240, 200),
    useSVG=True,
)

## 6. Your turn

Paste any SMILES (PubChem lists one for every compound) and re-run the cell.
Need another package? `conda install <pkg>` in a terminal, then *Kernel → Restart Kernel*.

In [ ]:
label, mol = to_mol("CN1C=NC2=C1C(=O)N(C(=O)N2C)C")   # caffeine; try "C1=CC=C2C(=C1)C=CC=C2" (naphthalene)
display(mol)
pd.Series(properties(mol), name=label)